<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/t81_558_class_02_5_beyond_cpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# T81-558: Applications of Deep Neural Networks

**Module 2: PyTorch for Neural Networks**

* Instructor: [Jeff Heaton](https://sites.washu.edu/jeffheaton/), McKelvey School of Engineering, [Washington University in St. Louis](https://engineering.washu.edu/index.html)
* For more information visit the [class website](https://sites.washu.edu/jeffheaton/t81-558/).

# Module 2 Material

* Part 2.1: Numeric Processing with PyTorch [[Video]]() [[Notebook]](t81_558_class_02_1_pytorch_numerical.ipynb)
* Part 2.2: Deep Learning with PyTorch [[Video]]() [[Notebook]](t81_558_class_02_2_pytorch_neural.ipynb)
* Part 2.3: Preprocessing for PyTorch [[Video]]() [[Notebook]](t81_558_class_02_3_feature_encode.ipynb)
* Part 2.4: nn.Module vs nn.Sequential for PyTorch [[Video]]() [[Notebook]](t81_558_class_02_4_pytorch_class_sequence.ipynb)
* **Part 2.5: Beyond the CPU** [[Video]]() [[Notebook]](t81_558_class_02_5_beyond_cpu.ipynb)

# Google CoLab Instructions

The following code ensures that Google CoLab is running the correct version of TensorFlow.

In [1]:
# Detect Colab if present
try:
    from google.colab import drive
    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

Note: using Google CoLab


# Part 2.5: Beyond the CPU

This section introduces hardware acceleration beyond the CPU. In PyTorch, accelerators fall into two broad categories:

- **General-purpose accelerators** - Devices such as CPUs, NVIDIA GPUs (CUDA), and Apple Silicon GPUs (MPS) that share a common programming model and can typically be used interchangeably with minimal code changes.
- **Specialized accelerators** - Devices such as Tensor Processing Units (TPUs) that require a different execution model and explicit support in code.

We begin with general-purpose accelerators.

## General-Purpose Accelerators

In PyTorch, most day-to-day work does not require specialized hardware like TPUs. Instead, it relies on a group of devices that share a common programming model and can be used with minimal code changes. These are referred to as *general-purpose accelerators*. In practice, this means that the same model and tensors can typically run on any of these devices by simply changing the device assignment.

The three most important general-purpose compute targets are:

- **CPU** - The default execution device. Always available, easy to debug, and sufficient for smaller models or initial development.
- **NVIDIA GPU (CUDA)** - The most widely used acceleration platform for deep learning. Provides significant speedup for training and inference, with strong support across PyTorch and the broader ecosystem.
- **Apple Silicon GPU (MPS)** - Available on Apple hardware. Supported by PyTorch through Metal Performance Shaders, allowing many models to run with minimal changes similar to CUDA.

These devices are unified in PyTorch through a consistent interface. In most cases, switching between them requires only moving the model and tensors to the appropriate device.

AMD GPUs also fit into this general category of accelerators. PyTorch supports AMD hardware through the ROCm platform, which provides functionality similar to CUDA. However, ROCm is less widely used and has more limited ecosystem support compared to NVIDIA GPUs. For this reason, AMD GPUs are beyond the scope of this course, but they represent a viable alternative in environments where ROCm is available.

The following code demonstrates how to detect which of these devices is available and select the best option automatically.




In [2]:
# Detect available hardware (TPU, CUDA, MPS, CPU)

device = "cpu"
tpu_available = False

# --- TPU detection (PyTorch/XLA) ---
try:
    import torch_xla.core.xla_model as xm

    device = xm.xla_device()
    tpu_available = True
    print(f"Using TPU: {device}")

except ImportError:
    # --- GPU / MPS / CPU fallback ---
    import torch

    if torch.cuda.is_available():
        device = torch.device("cuda")
        print(f"Using CUDA GPU: {torch.cuda.get_device_name(0)}")

    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Using Apple MPS GPU")

    else:
        device = torch.device("cpu")
        print("Using CPU")

# Optional: flag for downstream logic
print(f"TPU available: {tpu_available}")

Using CUDA GPU: NVIDIA A100-SXM4-80GB
TPU available: False


## Running Auto MPG on a General-Purpose Accelerator

In this section, we will run the Auto MPG regression model using a general-purpose compute device. These include CPUs, NVIDIA GPUs (CUDA), and Apple Silicon GPUs (MPS). All three are supported through the same PyTorch interface and can typically be used with only minor code changes.

The example that follows uses the same dataset, preprocessing steps, and neural network architecture introduced earlier. The goal is not to introduce new modeling concepts, but to demonstrate how PyTorch can automatically take advantage of available hardware acceleration.

The code performs the following steps:
- Detects the best available device (CPU, CUDA GPU, or MPS GPU)
- Loads and preprocesses the Auto MPG dataset
- Converts the data into PyTorch tensors on the selected device
- Defines and trains a neural network model
- Reports training loss during execution

The key idea is that the model and data are moved to a device using a consistent pattern. Once placed on the selected device, all computations occur there without further modification.

The following example demonstrates how to detect the available device and train the model using a general-purpose accelerator.

In [3]:
# Auto MPG regression on a general-purpose PyTorch device:
# CPU, NVIDIA GPU (CUDA), or Apple Silicon GPU (MPS)

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler

# --- Device setup ---
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using CUDA GPU: {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple Silicon GPU: MPS")
else:
    device = torch.device("cpu")
    print("Using CPU")

# --- Model ---
class Net(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.layer1 = nn.Linear(input_dim, 50)
        self.layer2 = nn.Linear(50, 25)
        self.layer3 = nn.Linear(25, output_dim)

    def forward(self, x):
        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        return self.layer3(x)

# --- Data ---
df = pd.read_csv(
    "https://data.heatonresearch.com/data/t81-558/auto-mpg.csv",
    na_values=["NA", "?"],
)

df["horsepower"] = df["horsepower"].fillna(df["horsepower"].median())
df = pd.get_dummies(df, columns=["origin"], dtype=float)

x_np = df[
    [
        "cylinders",
        "displacement",
        "horsepower",
        "weight",
        "acceleration",
        "year",
        "origin_1",
        "origin_2",
        "origin_3",
    ]
].values

y_np = df["mpg"].values

scaler = StandardScaler()
x_np = scaler.fit_transform(x_np)

x = torch.tensor(x_np, device=device, dtype=torch.float32)
y = torch.tensor(y_np, device=device, dtype=torch.float32)

# --- Training setup ---
model = Net(x.shape[1], 1).to(device)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# --- Training loop ---
for epoch in range(200):
    optimizer.zero_grad()

    outputs = model(x).flatten()
    loss = loss_fn(outputs, y)

    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, loss: {loss.item():.4f}")

Using CUDA GPU: NVIDIA A100-SXM4-80GB
Epoch 0, loss: 613.3287
Epoch 50, loss: 13.2808
Epoch 100, loss: 7.3420
Epoch 150, loss: 6.6403


## Tensor Processing Units (TPUs)

This book focuses primarily on NVIDIA Graphics Processing Units (GPUs) for deep learning acceleration. While NVIDIA GPUs are not the only option, they remain the most widely supported and commonly used hardware for deep learning, particularly with PyTorch.

Alternative hardware platforms do exist. AMD GPUs are supported through ROCm, and Intel provides GPU acceleration through oneAPI, though both ecosystems are less widely adopted for deep learning training compared to NVIDIA CUDA. Apple silicon systems also provide GPU acceleration through Metal (MPS), which is supported by PyTorch and allows many models to run with minimal code changes.

Google offers Tensor Processing Units (TPUs), which are specialized AI accelerator chips available through its cloud platform. TPUs are application-specific integrated circuits (ASICs) designed to accelerate neural network workloads. Originally developed for TensorFlow, TPUs are now supported in PyTorch through the PyTorch/XLA interface.

However, TPUs are not a drop-in replacement in the same way as NVIDIA CUDA or Apple MPS devices. In PyTorch, moving between CPU, CUDA, and MPS devices typically requires only minor changes, primarily related to device placement. TPUs, by contrast, require a different execution model, additional libraries, and more specialized code. As a result, they are generally only used when explicitly targeted, rather than as a default acceleration option.

Because of their broad availability, ease of use, and strong ecosystem support, this course focuses on NVIDIA GPUs. We will briefly examine TPUs for completeness, but they are not required for this course.

We will begin by checking the environment to determine which hardware (CPU, GPU, or TPU) is available.

## Running Auto MPG on a TPU

So far, all examples in this chapter have used CPUs or GPUs. In this section, we will run the same Auto MPG regression model on a Tensor Processing Unit (TPU). This provides a useful comparison because the model, dataset, and preprocessing steps remain unchanged.

TPUs are specialized accelerators developed by :contentReference[oaicite:0]{index=0} for neural network workloads. Unlike GPUs, TPUs are not a general-purpose, drop-in device for PyTorch. GPUs (CUDA) and Apple Silicon (MPS) typically require only minor changes, such as moving tensors and models to a device. TPUs, however, require explicit support through the PyTorch/XLA library and follow a different execution model.

Because of this, running on a TPU introduces a few additional steps:
- Importing and initializing the PyTorch/XLA runtime
- Selecting a TPU device using `xm.xla_device()`
- Using a TPU-specific optimizer step (`xm.optimizer_step`) instead of the standard `optimizer.step()`

The example below reuses the Auto MPG dataset and neural network from earlier sections. The goal is not to introduce new modeling concepts, but to highlight the differences required to execute the same code on a TPU.

If you are using :contentReference[oaicite:1]{index=1}, you can enable a TPU by selecting:

**Runtime → Change runtime type → TPU**

If a TPU is not available, the example will fall back to CPU or GPU execution.

The most important takeaway is that TPUs must be explicitly targeted. If you did not intentionally configure your code to use a TPU, you are almost certainly not using one.

In [1]:
# Auto MPG regression on TPU (PyTorch/XLA)
# In Colab: Runtime > Change runtime type > TPU

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler

# --- TPU setup ---
try:
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
    use_tpu = True
    print(f"Using TPU: {device}")
except ImportError:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_tpu = False
    print(f"Using fallback device: {device}")

# --- Model ---
class Net(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.layer1 = nn.Linear(input_dim, 50)
        self.layer2 = nn.Linear(50, 25)
        self.layer3 = nn.Linear(25, output_dim)

    def forward(self, x):
        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        return self.layer3(x)

# --- Data ---
df = pd.read_csv(
    "https://data.heatonresearch.com/data/t81-558/auto-mpg.csv",
    na_values=["NA", "?"],
)

df["horsepower"] = df["horsepower"].fillna(df["horsepower"].median())
df = pd.get_dummies(df, columns=["origin"], dtype=float)

x_np = df[
    [
        "cylinders",
        "displacement",
        "horsepower",
        "weight",
        "acceleration",
        "year",
        "origin_1",
        "origin_2",
        "origin_3",
    ]
].values

y_np = df["mpg"].values

scaler = StandardScaler()
x_np = scaler.fit_transform(x_np)

x = torch.tensor(x_np, device=device, dtype=torch.float32)
y = torch.tensor(y_np, device=device, dtype=torch.float32)

# --- Training setup ---
model = Net(x.shape[1], 1).to(device)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# --- Training loop ---
for epoch in range(200):
    optimizer.zero_grad()

    outputs = model(x).flatten()
    loss = loss_fn(outputs, y)

    loss.backward()

    # TPU requires xm.optimizer_step
    if use_tpu:
        xm.optimizer_step(optimizer)
    else:
        optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, loss: {loss.item():.4f}")

/tmp/ipykernel_1277/2220608138.py:13: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()


Using TPU: xla:0
Epoch 0, loss: 602.0582
Epoch 50, loss: 12.2517
Epoch 100, loss: 7.4582
Epoch 150, loss: 6.6859
